In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 27 
const aa = 38
const N  = 1462439
const I0 = 1
const S0 = 1316195
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [3, 7, 10, 4, 14, 35, 92, 98, 216, 374, 434, 417, 447, 275, 151, 83, 67, 48, 37, 29, 42, 46, 53, 73, 87, 111, 123, 124, 99, 135, 106, 74, 39, 13]


tau = length(Istar_obs)

model_tag_sym = :powerlaw

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 2

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [powerlaw_model] Fitting chain 2 (tau=34)
[ Info: [powerlaw] iter 1000/1000000 elapsed=3.7s, rate=0.113, mean=[1.857, 0.00091, 1.137, 0.434], std=[0.2821, 0.000384, 0.0117, 0.0286] [ADAPT]
[ Info: [powerlaw] iter 2000/1000000 elapsed=6.9s, rate=0.089, mean=[2.007, 0.00078, 1.166, 0.466], std=[0.2462, 0.000302, 0.0371, 0.0418] [ADAPT]
[ Info: [powerlaw] iter 3000/1000000 elapsed=9.2s, rate=0.082, mean=[2.051, 0.00074, 1.208, 0.483], std=[0.2113, 0.000258, 0.0634, 0.0428] [ADAPT]
[ Info: [powerlaw] iter 4000/1000000 elapsed=11.4s, rate=0.083, mean=[2.090, 0.00073, 1.252, 0.485], std=[0.2017, 0.000230, 0.0886, 0.0384] [ADAPT]
[ Info: [powerlaw] iter 5000/1000000 elapsed=13.6s, rate=0.082, mean=[2.102, 0.00072, 1.289, 0.488], std=[0.1836, 0.000210, 0.1072, 0.0366] [ADAPT]
[ Info: [powerlaw] iter 6000/1000000 elapsed=15.8s, rate=0.082, mean=[2.127, 0.00072, 1.338, 0.491], std=[0.1759, 0.000196, 0.1399, 0.0344] [ADAPT]
[ Info: [powerlaw] iter 7000/1000000 elapsed=18.1s, rate=0.082, m